# Daily Profile Generation


This notebook converts 15-minute electricity demand data into daily profiles with 96 quarter-hour demand points.

## Steps:
1. Load cleaned data from data/processed/cleaned_data.csv
2. Create time_slot from Hour and Minute (0-95)
3. Pivot each Date into p00 to p95 columns
4. Remove incomplete days
5. Calculate daily statistics (mean, peak, min, std, etc.)
6. Add temporal features
7. Save daily profiles to data/processed/daily_profiles.csv


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Set up paths
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CLEANED_DATA_PATH = DATA_PROCESSED_DIR / "cleaned_data.csv"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")


In [ ]:

# Load cleaned data
if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned data not found at {CLEANED_DATA_PATH}")

df_clean = pd.read_csv(CLEANED_DATA_PATH)

print(f"Cleaned data loaded successfully!")
print(f"Shape: {df_clean.shape}")
print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
print(f"Columns: {list(df_clean.columns)}")
print(f"\nFirst few rows:")
print(df_clean.head())


In [ ]:

# Create time_slot from Hour and Minute (0-95 for 15-minute intervals)
df_clean['time_slot'] = df_clean['Hour'] * 4 + df_clean['Minute'] // 15

print(f"Time slots created: {df_clean['time_slot'].min()} to {df_clean['time_slot'].max()}")
print(f"Unique time slots: {sorted(df_clean['time_slot'].unique())}")


In [ ]:

# Get demand column name (assuming it's the second column after timestamp)
demand_col = df_clean.columns[1]  # Usually the demand/load column
timestamp_col = df_clean.columns[0]  # Usually the timestamp column

print(f"Using demand column: {demand_col}")
print(f"Using timestamp column: {timestamp_col}")

# Pivot to create daily profiles
daily_profiles = df_clean.pivot_table(
    index='Date',
    columns='time_slot',
    values=demand_col,
    aggfunc='mean'
)

# Rename columns to p00, p01, ..., p95
daily_profiles.columns = [f"p{str(col).zfill(2)}" for col in daily_profiles.columns]

print(f"Daily profiles created: {daily_profiles.shape}")
print(f"Profile columns: {list(daily_profiles.columns)}")
print(f"\nFirst few rows:")
print(daily_profiles.head())


In [ ]:

# Remove days with missing time slots (incomplete days)
complete_days = daily_profiles.notna().all(axis=1)
daily_profiles = daily_profiles[complete_days]

print(f"Complete days: {complete_days.sum()}/{len(daily_profiles)}")
print(f"Removed {len(daily_profiles) - complete_days.sum()} incomplete days")
print(f"Final daily profiles: {daily_profiles.shape}")

# Fill any remaining missing values with interpolation
daily_profiles = daily_profiles.interpolate(method='linear', axis=1)
daily_profiles = daily_profiles.fillna(method='bfill', axis=1).fillna(method='ffill', axis=1)


In [ ]:

# Calculate daily statistics
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]

daily_profiles['daily_mean'] = daily_profiles[profile_cols].mean(axis=1)
daily_profiles['daily_peak'] = daily_profiles[profile_cols].max(axis=1)
daily_profiles['daily_min'] = daily_profiles[profile_cols].min(axis=1)
daily_profiles['daily_std'] = daily_profiles[profile_cols].std(axis=1)
daily_profiles['total_demand'] = daily_profiles[profile_cols].sum(axis=1)

# Find peak time slot
daily_profiles['peak_time_slot'] = daily_profiles[profile_cols].idxmax(axis=1)
daily_profiles['peak_time_slot'] = daily_profiles['peak_time_slot'].str.replace('p', '').astype(int)

print("Daily statistics calculated:")
print(f"- Mean demand: {daily_profiles['daily_mean'].mean():.2f} kW")
print(f"- Peak demand: {daily_profiles['daily_peak'].mean():.2f} kW")
print(f"- Std demand: {daily_profiles['daily_std'].mean():.2f} kW")


In [ ]:

# Convert Date to datetime and add temporal features
daily_profiles.index = pd.to_datetime(daily_profiles.index)
daily_profiles['date'] = daily_profiles.index.strftime('%Y-%m-%d')
daily_profiles['day_of_week'] = daily_profiles.index.dayofweek
daily_profiles['month'] = daily_profiles.index.month
daily_profiles['is_weekend'] = (daily_profiles.index.dayofweek.isin([5, 6])).astype(int)

# Reorder columns to have date first, then stats, then profiles
base_cols = ['date', 'daily_mean', 'daily_peak', 'daily_min', 'daily_std', 
            'total_demand', 'peak_time_slot', 'day_of_week', 'month', 'is_weekend']
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]
all_cols = base_cols + profile_cols
daily_profiles = daily_profiles[all_cols]

print("Temporal features added:")
print(f"- Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")
print(f"- Days of week: {sorted(daily_profiles['day_of_week'].unique())}")
print(f"- Months: {sorted(daily_profiles['month'].unique())}")
print(f"- Weekend days: {daily_profiles['is_weekend'].sum()}")


In [ ]:

# Save daily profiles
daily_profiles.to_csv(DAILY_PROFILES_PATH, index=False)

print(f"Daily profiles saved to: {DAILY_PROFILES_PATH}")
print(f"Final shape: {daily_profiles.shape}")
print(f"File size: {DAILY_PROFILES_PATH.stat().st_size / 1024 / 1024:.1f} MB")

print("\nDaily profiles summary:")
print(f"- Total profiles: {len(daily_profiles)}")
print(f"- Profile columns: {len([col for col in daily_profiles.columns if col.startswith('p')])}")
print(f"- Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")

print("\nSample of final data:")
print(daily_profiles[['date', 'daily_mean', 'daily_peak', 'daily_peak']].head())
